# Monthly Anomalies and Arctic Climate Stripes

**Yiwen Li (YL) · 20-minute online seminar · CDO/NCO processing + NCL graphics**

This Binder demonstration reproduces [EG's shared Python notebook](https://github.com/egalytska/Fresh_eyes/blob/393cfcf876fa47c6c231c3be70dbd9f8552fe3a3/stripes_EG.ipynb), based on Abhnil's workshop.

**Same task across languages:** 1981–2010 monthly climatology → 1979–2014 monthly anomalies → Robinson map → 65–90°N monthly line → 36 annual Arctic stripes.

> Use CDO/NCO to prepare the data. Use Python to understand the data. Use NCL to render the three figures.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import json

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import xarray as xr
from IPython.display import Image, display

DATA = Path(os.environ.get('WORKSHOP_INPUT', 'data/demo/tas_demo.nc'))
ROOT = Path('outputs/yl_monthly')
assert DATA.is_file(), f'Missing input: {DATA}. Run make demo-data or provide WORKSHOP_INPUT.'
for tool in ('cdo', 'ncks', 'ncwa', 'ncl'):
    assert shutil.which(tool), f'Missing tool: {tool}'
with xr.open_dataset(DATA) as ds:
    print('Input:', DATA)
    print('Dataset:', ds.attrs.get('title', ds.attrs.get('source_id', 'Inspect provenance')))
    print('Summary:', ds.attrs.get('summary', 'Use the original CMIP data citation.'))
    synthetic = ds.attrs.get('source_id') == 'WORKSHOP-DEMO'
print('SYNTHETIC teaching data; not CMIP model results.' if synthetic else 'External data: check model, member, calendar and grid against EG.')

## 0:00–2:00 | The same three figures, in another toolchain

The default Binder input is a deterministic **synthetic teaching dataset**, with the same monthly variable structure and time span as the original ACCESS-CM2 file. It demonstrates the calculation, not actual model warming. Full ACCESS-CM2 data can be supplied via `WORKSHOP_INPUT` for an identical-data comparison; a spatially strided sample will produce different regional values.

**Chat prompt:** What does a monthly anomaly compare January with?

In [ ]:
preview = Path('outputs/demo/monthly/03_arctic_annual_stripes.png')
if preview.exists() and synthetic:
    display(Image(filename=str(preview), width=900))
    print('Preview: synthetic Arctic annual anomalies, 1979–2014; baseline 1981–2010.')
else:
    print('The three figures will be generated below.')

## 2:00–4:00 | Inspect variable, calendar, period and grid

EG uses ACCESS-CM2 historical `r1i1p1f1`, Amon `tas`, grid `gn`, version `v20191108`, 1850–2014. The tutorial analyses 1979–2014 and builds its baseline from 1981–2010. Confirm all baseline months are available before selecting the analysis period.

In [ ]:
summary = subprocess.run(['cdo', '-s', 'sinfo', str(DATA)], check=True, capture_output=True, text=True)
print(summary.stdout)
with xr.open_dataset(DATA) as ds:
    display(ds.tas)
    assert int(ds.time.dt.year.min()) <= 1981 and int(ds.time.dt.year.max()) >= 2014

## 4:00–6:30 | Python reference: follow EG's definitions

A monthly anomaly subtracts the **matching calendar month's** 1981–2010 climatology at each grid cell. The Arctic series is the simple arithmetic mean of grid points at 65–90°N. Area weighting would change EG's calculation.

The annual mean gives each monthly sample equal weight, as in the reference notebook.

In [ ]:
with xr.open_dataset(DATA) as ds:
    tas = (ds.tas - 273.15).load()
tas_ref = tas.sel(time=slice('1981-01-01', '2010-12-31'))
tas_clim = tas_ref.groupby('time.month').mean('time')
tas_anom = tas.groupby('time.month') - tas_clim
tas_anom = tas_anom.sel(time=slice('1979-01-01', '2014-12-31'))
mean_map = tas_anom.mean('time')
arctic = tas_anom.where((tas_anom.lat >= 65) & (tas_anom.lat <= 90), drop=True)
arctic_monthly = arctic.mean(('lat', 'lon'))
arctic_annual = arctic_monthly.groupby('time.year').mean('time')
print(f'{tas_clim.sizes["month"]} climatology months; {tas_anom.sizes["time"]} anomaly months; {arctic_annual.size} annual stripes')

## 6:30–11:00 | CDO/NCO: the same operations

| Python task | CDO/NCO operation |
|---|---|
| `ds.tas` | `ncks -v tas` |
| Kelvin → Celsius | `cdo subc,273.15` |
| 1981–2010 monthly climatology | `cdo ymonmean -selyear,1981/2010` |
| Subtract matching climatology month | `cdo ymonsub` |
| Retain 1979–2014 | `cdo seldate,1979-01-01,2014-12-31` |
| Mean anomaly map | `cdo timmean` |
| Select 65–90°N | `ncks -d lat,65.0,90.0` |
| Unweighted Arctic grid-point mean | `ncwa -a lat,lon` |
| Annual mean | `cdo yearmean` |

Read CDO chains from right to left. Show the prepared script, then run it; do not type the full workflow during the talk.

In [ ]:
print(Path('scripts/run_cdo_nco_monthly_anomaly.sh').read_text())

In [ ]:
subprocess.run(['bash', 'scripts/run_cdo_nco_monthly_anomaly.sh', str(DATA), str(ROOT / 'cdo_nco')], check=True)

## 11:00–13:00 | Verify all four scientific outputs

Compare every monthly grid-cell anomaly, the mean map, the Arctic monthly series, and the Arctic annual series. Check 432 months, 36 years, and a near-zero baseline for each calendar month. A visually similar plot alone is insufficient.

In [ ]:
subprocess.run(['python', 'scripts/run_python_monthly_anomaly.py', str(DATA), str(ROOT / 'python')], check=True)
subprocess.run(['python', 'scripts/validate_monthly_anomaly.py', '--cdo-nco', str(ROOT / 'cdo_nco'), '--python', str(ROOT / 'python'), '--report', str(ROOT / 'validation.json')], check=True)
report = json.loads((ROOT / 'validation.json').read_text())
assert all(item['passed'] for item in report.values())
print('PASS: independent CDO/NCO and xarray results agree.')

## 13:00–17:00 | NCL renders the three reference figures

NCL reads the processed NetCDF files. The Robinson map uses `coolwarm` and the shared −0.2 to +0.4 °C range. The line shows all 432 monthly Arctic anomalies. The stripes show one equal-width rectangle per year using blue–white–red and ±max(|annual anomaly|).

**Display convention:** NCL fonts and coastlines may differ from matplotlib/cartopy. The scientific arrays and colour normalization are the comparison target.

In [ ]:
subprocess.run(['ncl', f'input_dir="{ROOT / "cdo_nco"}"', f'output_dir="{ROOT}"', 'scripts/make_eg_figures.ncl'], check=True)
subprocess.run(['python', 'scripts/check_ncl_outputs.py', str(ROOT)], check=True)

In [ ]:
display(Image(filename=str(ROOT / '01_mean_anomaly_map.png'), width=900))
print('Mean of 1979–2014 monthly anomalies relative to 1981–2010 monthly climatology.')

In [ ]:
display(Image(filename=str(ROOT / '02_arctic_monthly_anomaly.png'), width=900))
print('65–90°N arithmetic grid-point mean, 432 monthly values; no area weighting.')

In [ ]:
display(Image(filename=str(ROOT / '03_arctic_annual_stripes.png'), width=900))
print('Arctic annual stripes: 1979–2014, 36 years, equal monthly weights.')
print(f'Symmetric colour limit: ±{float(abs(arctic_annual).max()):.4f} °C')

## 17:00–19:00 | Predict, then test in the hands-on session

1. What changes if the baseline is 1981–2000? Both Python and CDO must change the **same** baseline.
2. What changes if an Arctic area-weighted mean replaces the arithmetic mean? Treat that as an extension and change both implementations.
3. Why can annual averaging still show warming after removing the seasonal cycle?

Keep these as predictions during the 20-minute talk. Use `notebooks/04_monthly_anomaly_hands_on.ipynb` in the shared hands-on block. Binder sessions are temporary: download your notebook/results before leaving.

## 19:00–20:00 | Tool choice and handoff

CDO expresses monthly climatology and anomalies concisely. NCO preserves the reference's exact Arctic mean and edits metadata. Python checks and explains the results. NCL renders the three climate graphics.

**Takeaway:** Use CDO/NCO to prepare the data. Use Python to understand the data.

Source audit and exact mapping: [docs/EG_SHARED_NOTEBOOK_MAPPING.md](../docs/EG_SHARED_NOTEBOOK_MAPPING.md).